# 04. Evaluation, Comparative Analysis & Inference
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
### Academic Research Project: **GeoFinBERT**

---
### Evaluation Protocol:
- **Partition**: Strict unseen chronological test split (15% of human-labeled corpus).
- **Zero Leakage**: No test period text was present in Stage 1 DAPT or Stage 2 training.
- **Metrics**: Accuracy, Macro F1, Weighted F1, Macro Precision, Macro Recall, Per-Class F1, Confusion Matrix.
- **Inference Pipeline**: Interactive deployment for live market intelligence and geopolitical risk tracking.


In [ ]:
# Setup & Imports
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Environment Detected] Installing dependencies...")
    !pip install -q -r requirements.txt
    !pip install -q -e .

import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath("src"))

from geofinbert.config import load_config
from geofinbert.models.classifier import GeoFinBERTForSequenceClassification
from geofinbert.data.dataset import SentimentClassificationDataset
from geofinbert.training.trainer_classifier import evaluate_classifier
from geofinbert.evaluation.metrics import compute_classification_metrics, format_metrics_table
from geofinbert.inference import GeoFinBERTInferencePipeline

config = load_config("configs/config.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation Device:", device)


## Step 1: Load Test Partition & Models


In [ ]:
test_path = os.path.join(config["data"]["splits_dir"], "sentiment_test.parquet")
test_df = pd.read_parquet(test_path)
print(f"Loaded Test Set: {len(test_df):,} samples.")

baseline_dir = config["stage2_finetuning"]["baseline_output_dir"]
adapted_dir = config["stage2_finetuning"]["adapted_output_dir"]

from transformers import BertTokenizer
tok = BertTokenizer.from_pretrained(adapted_dir if os.path.exists(adapted_dir) else "ProsusAI/finbert")
test_ds = SentimentClassificationDataset(test_df["text"].tolist(), test_df["label"].tolist(), tok, max_length=128)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Load Models
print("Loading Baseline FinBERT...")
b_model = GeoFinBERTForSequenceClassification.from_pretrained(baseline_dir).to(device)
_, b_metrics = evaluate_classifier(b_model, test_loader, device)

print("Loading GeoFinBERT...")
a_model = GeoFinBERTForSequenceClassification.from_pretrained(adapted_dir).to(device)
_, a_metrics = evaluate_classifier(a_model, test_loader, device)


## Step 2: Head-to-Head Academic Metrics Table


In [ ]:
from IPython.display import display, Markdown

table_md = format_metrics_table(b_metrics, a_metrics)
display(Markdown(table_md))


## Step 3: Confusion Matrix Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_labels = ["Positive", "Negative", "Neutral"]

# Baseline FinBERT
sns.heatmap(
    b_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    ax=axes[0]
)
axes[0].set_title(f"Baseline FinBERT (Acc: {b_metrics['accuracy']:.2%}, Macro F1: {b_metrics['macro_f1']:.4f})")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Ground Truth")

# GeoFinBERT
sns.heatmap(
    a_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=class_labels,
    yticklabels=class_labels,
    ax=axes[1]
)
axes[1].set_title(f"GeoFinBERT (Acc: {a_metrics['accuracy']:.2%}, Macro F1: {a_metrics['macro_f1']:.4f})")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Ground Truth")

plt.tight_layout()
plt.show()


## Step 4: Per-Class F1 & Precision/Recall Breakdown


In [ ]:
classes = ["Positive", "Negative", "Neutral"]
b_f1 = [b_metrics["per_class"][c.lower()]["f1_score"] for c in classes]
a_f1 = [a_metrics["per_class"][c.lower()]["f1_score"] for c in classes]

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, b_f1, width, label="Baseline FinBERT", color="#e74c3c")
ax.bar(x + width/2, a_f1, width, label="GeoFinBERT (Adapted)", color="#2ecc71")

ax.set_ylabel("F1 Score")
ax.set_title("Per-Class Sentiment F1 Comparison")
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Step 5: Live Interactive Inference Pipeline
Test GeoFinBERT on novel geopolitical events and energy market headlines.


In [ ]:
# Initialize Inference Pipeline
pipeline = GeoFinBERTInferencePipeline(model_dir_or_name=adapted_dir, device="auto")

sample_headlines = [
    "Crude oil surges 4% following missile strikes on Red Sea commercial oil tankers.",
    "Iran and IAEA resume diplomatic dialogue regarding uranium enrichment monitoring in Vienna.",
    "Gold spot prices tumble 2.5% as bullion demand weakens amid rising treasury yields.",
    "OPEC+ ministers agree on surprise 500,000 barrel per day voluntary crude production cut.",
    "US Department of Treasury imposes fresh secondary sanctions on foreign entities facilitating Iranian petroleum exports.",
    "Central banks worldwide accelerate sovereign bullion purchases to diversify reserves away from dollar holdings."
]

results = pipeline.predict_batch(sample_headlines)

for r in results:
    print(f"\nHeadline: {r['text']}")
    print(f"Sentiment: {r['prediction'].upper()} (Confidence: {r['confidence']:.2%})")
    print(f"Probabilities: {r['probabilities']}")


## Step 6: Model Export & Archiving
GeoFinBERT weights, tokenizer configurations, and metrics are finalized and ready for research publication.
